# Post process data extracted using LLMs
1. Load data
2. Convert to correct format (numeric for numerical columns)
3. Eventually explode lists for geocoding?
4. Geocoding
5. Sanity checks

In [15]:
import numpy as np
import pandas as pd
import geopandas as gpd
import geopy as gpy
import time
import itertools
from matplotlib import pyplot as plt
from src.data import *
from src.plot_functions import *
from src.post_process_functions import *
from src.geocoding import *



In [18]:
#load data
model_name = "meta-llama/llama-4-scout-17b-16e-instruct"
nreports = 50
res_savename = f"llm_response_hazmain_nb_format_std_units_impact_{nreports}rep_test_{model_name.replace('/', '_')}.csv"
response_df = pd.read_csv(DATA_OUT_LLMS+res_savename)

In [19]:
#get rid of nans
response_df = response_df.dropna(subset=["nathaz_text"])

In [20]:
#convert numerical columns
num_cols = ["impactValue", "startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"]
response_df_proc = cp.deepcopy(response_df)
response_df_proc = format_output(response_df_proc, num_cols=num_cols)


In [37]:
#add iso3
response_df_proc["country_iso3"] = response_df_proc["country"].apply(country_name_to_iso3)
response_df_proc["country_iso3_kw"] = response_df_proc["country_kw"].apply(country_name_to_iso3)

In [22]:
#format units
from spacy.lang.en import English
from spacy.lang.punctuation import TOKENIZER_PREFIXES, TOKENIZER_SUFFIXES, TOKENIZER_INFIXES
from spacy.lang.en import TOKENIZER_EXCEPTIONS
from spacy.tokenizer import Tokenizer
from spacy.util import compile_prefix_regex, compile_suffix_regex, compile_infix_regex

## Sanity checks

In [23]:
response_df_proc.impactUnit.value_counts()

impactUnit
people                             160
houses                              22
kilometer**2                        18
homes                               15
families                            12
heads                                7
person                               3
households                           3
schools                              3
cases                                2
livestock                            2
hospitals                            2
buildings                            2
latrines                             2
kilometer²                           2
km                                   2
cows and sheep                       1
health institutions                  1
institutions                         1
health centers                       1
establishments                       1
facilities                           1
communities                          1
water sources                        1
properties                           1
children      

In [29]:
response_df_proc.impactType.value_counts()

impactType
Affected People                        77
Agriculture                            46
Residential Buildings                  42
Displaced People                       37
Human Deaths                           37
WASH infrastructure                    28
Blocked roads                          20
Healthcare Infrastructure              20
Education Infrastructure               18
Injured People                         17
Homeless People                        11
Missing People                          8
IT and Communication Infrastructure     7
Livestock                               7
Cholera cases                           2
Informal Settlements                    1
Name: count, dtype: int64

In [31]:
response_df_proc.hazards.value_counts()

hazards
['Flood']                                   201
['Earthquake']                               40
['Drought']                                  21
['Tropical storm']                           19
['Flood', 'Convective Storm']                19
['Extreme cold temperature']                 12
['Convective Storm']                         12
['Volcanic activity', 'Earthquake']           9
['Wildfire', 'Extreme warm temperature']      8
['Flood', 'Mass movement']                    8
['Wildfire']                                  5
['Extreme warm temperature']                  4
['Fire']                                      4
['Volcanic activity']                         4
['Mass movement', 'Flood']                    4
['Flood', 'Tropical storm']                   3
['Cholera']                                   2
['Cholera', 'Flood']                          1
['Extreme warm temperature', 'Drought']       1
['Wildfire', 'Drought']                       1
Name: count, dtype: int64

In [38]:

def format_number(num):
    """
    Formats a number into a string, removing unnecessary trailing zeros
    and the decimal point if it's not needed.
    """
    if isinstance(num, float) and num.is_integer():
        # If the number is a float but represents an integer
        return str(int(num))
    return str(num).rstrip('0').rstrip('.') if '.' in str(num) else str(num)
def value_in_text(extract_df):
    extract_df["value_in_text"] = extract_df.apply(lambda x: format_number(x["impactValue"]) in "".join(x["nathaz_text"]), axis=1)
    return extract_df

response_df_proc = value_in_text(response_df_proc)

In [39]:
response_df_proc.value_in_text.value_counts()

value_in_text
True     333
False     45
Name: count, dtype: int64

In [41]:
country_pop = pd.read_csv(DATA_PATH +"API_SP.POP.TOTL_DS2_en_csv_v2_131993/"+"API_SP.POP.TOTL_DS2_en_csv_v2_131993.csv",sep=',', header=2)
country_pop = country_pop.dropna(how="all",axis=1)
def pop_cntry_check(extracted_data, country_pop):
    def check_pop(x):
        year = str(pd.to_datetime(x["reportDate"]).year)
        year_check = year if year in country_pop[country_pop["Country Code"] == x["country_iso3_kw"]].columns else "2023"
        pop_year = country_pop[country_pop["Country Code"] == x["country_iso3_kw"]][year_check].values[0]
        return x["impactValue"] < pop_year if x["impactUnit"] == "people" else np.nan
    extracted_data["pop_cntry_check"] = np.nan
    extracted_data["pop_cntry_check"] = extracted_data.apply(check_pop, axis=1)
    return extracted_data

response_df_proc = pop_cntry_check(response_df_proc, country_pop)

In [43]:
response_df_proc.pop_cntry_check.value_counts()

pop_cntry_check
True     159
False      1
Name: count, dtype: int64